# Introduction <a name="introduction"></a>

This notebook provides a comprehensive guide on utilizing the `watsonx.governance` Factsheets Python client to **create and manage models** and **AI use cases**. 

It illustrates the application of various **approaches** and **versioning** (Major, Minor, Patch) for effectively tracking models within an AI use case.

For detailed documentation on these features, please refer to the [IBM AI Factsheet documentation](https://s3.us.cloud-object-storage.appdomain.cloud/aifactsheets-client/index.html).


**Required Services:**
- `watsonx.governance`
- `watsonx.ai`

**Required Packages:**
- **IBM Facts Client Python SDK (>=1.0.101)**
- **IBM-watsonx-ai Python SDK**


  



In [ ]:
!pip install -U ibm-aigov-facts-client --quiet
!pip install -U ibm-watsonx-ai --quiet
!pip install -U python-dotenv --quiet
!pip install -U scikit-learn

In [ ]:
import warnings
import shutil
import os
from dotenv import load_dotenv
import os
from ibm_aigov_facts_client import AIGovFactsClient,CloudPakforDataConfig
from IPython.display import display, Markdown


shutil.rmtree('./mlruns', ignore_errors=True)
load_dotenv()



- Flag `run_cleanup_at_end` offers option to delete created assets at the end of the notebook.The notebook will show URL to UI for model and model use case at certain cells. By dafault we set it to `run_cleanup_at_end=False` so you can access UI and see the changes. If you decide to cleanup assets at the end, set `run_cleanup_at_end=True` and remember cells showing links to UI will `NOT` work in that case.

In [2]:
run_cleanup_at_end=True

- `Experiment` and `model names` are just for illustration purposes - they can be customized.

- Model container type can be `space` or `project`.

In [ ]:
experiment_name="IrisClassification"
MODEL_NAME="IrisScikitModel"
container_type="project"
container_id=os.getenv("CONTAINER_ID", "<if you can't use .env you can provide your value here>") # Project_id where the model has to store 

---
## Authentication Setup<a name="setup"></a>

### IBM watsonx.goverance software<a name="Watsonx.Gov-Platform"></a>
- Service url is the Cloud pak for data platform host URL. For skytap environment, it would be the internal nginx URL.
- You can either use user `password` or platform `apikey` to authenticate

In [ ]:

    
creds=CloudPakforDataConfig(service_url=os.getenv("CPD_SERVICE_URL", "<if you can't use .env you can provide your value here>"),
    username=os.getenv("CPD_USERNAME", "<if you can't use .env you can provide your value here>"),
    password=os.getenv("CPD_PASSWORD", "<if you can't use .env you can provide your value here>"))

   

## Client Initialization
- Container type would be either `space` or `project`. To use get/set environment utilities, model asset should be stored in Space.

- If running this notebook multiple times with same experiment name or anytime face error saying `Experiment with same name already exists`, use `set_as_current_experiment=True` when initiating client




In [5]:
facts_client = AIGovFactsClient(cloud_pak_for_data_configs=creds,experiment_name= experiment_name, container_type=container_type,container_id=container_id, set_as_current_experiment=True)

2026/03/19 12:46:46 INFO : Experiment IrisClassification does not exist, creating new experiment
2026/03/19 12:46:46 INFO : Experiment successfully created with ID 227558261068579785 and name IrisClassification
2026/03/19 12:46:46 INFO : Autolog enabled Successfully


---

## Create and Train Model <a name="#createmodel"></a>

- This sample code demonstrates creating and training a model, specifically a classifier.
- Model development is achieved without writing any IBM or watsonx.governance-specific code.
- Key training facts are automatically captured in the background and can be saved to a factsheet later.


In [6]:
import pandas as pd
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.metrics import accuracy_score
import numpy as np

# Get testdata for iris.
iris=datasets.load_iris()

x=iris.data
y=iris.target

# Split training and test data
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.5)

## Model Training

The following code will be used to train the model. During the training process, key facts and metrics will be automatically captured in the background by the `IBM watsonx.governance` factsheets client. These facts can later be saved and reviewed in a factsheet.

**Note:** Although there is no need to include any specific `IBM watsonx.governance` code for this process, you will still observe output


In [7]:
# Train model
classifier=tree.DecisionTreeClassifier()

#This is the main training method. No watxonx.governance code is directly needed but as you can see from the output the training details are logged
classifier.fit(x_train,y_train) 

# Predict model
predictions=classifier.predict(x_test)

# Check accuracy for the model
print(accuracy_score(y_test,predictions))

0.9466666666666667


---
## Store Model as a watsonx.ai / WML Asset in Project <a name="#savemodel"></a>

At present, the model exists solely as a Scikit-Learn object in memory. The subsequent step is to register it as a watsonx.ai / WML asset.

In [8]:
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials


Credentials = {
                   "url": creds.url,
                   "username": creds.username,
                   "password" : creds.password,
                   "instance_id": "openshift",
                   "version" : "5.3"
                  }


2026/03/19 12:46:49 INFO : logging results to factsheet for run_id e817e510aac74fe38c7e1a694116d04e


In [9]:
watsonx_ai_client = APIClient(Credentials)
watsonx_ai_client.version
watsonx_ai_client.set.default_project(container_id)

2026/03/19 12:46:52 INFO : Successfully logged results to Factsheet service for run_id e817e510aac74fe38c7e1a694116d04e under asset_id: fc31ce80-8848-4e12-9950-618f48021ea2 and project_id : eb58b8b3-2fd3-4702-944b-12299ec8424d


'SUCCESS'

### Define and Prepare Model Metadata for watsonx.ai / WML

The following code defines the software specification and model properties, and prepares the model metadata for registration with watsonx.ai / WML.


In [21]:
software_spec_uid = watsonx_ai_client.software_specifications.get_id_by_name("runtime-25.1-py3.12")
print("Software Specification ID: {}".format(software_spec_uid))

model_props = {
    watsonx_ai_client._models.ConfigurationMetaNames.NAME:"{}".format(MODEL_NAME),
    watsonx_ai_client._models.ConfigurationMetaNames.TYPE: "scikit-learn_1.6",
    watsonx_ai_client._models.ConfigurationMetaNames.SOFTWARE_SPEC_UID: software_spec_uid,
    watsonx_ai_client._models.ConfigurationMetaNames.LABEL_FIELD:"target",
}

facts_client.export_facts.prepare_model_meta(wml_client=watsonx_ai_client,meta_props=model_props)



Software Specification ID: f47ae1c3-198e-5718-b59d-2ea471561e9e


{'name': 'IrisScikitModel',
 'type': 'scikit-learn_1.6',
 'software_spec': 'f47ae1c3-198e-5718-b59d-2ea471561e9e',
 'label_column': 'target',
 'custom': {'experiment_id': '35d9b384f9184fdfb05cbe97372a7d97',
  'experiment_name': 'IrisClassification'}}

### Store Model and Retrieve Model ID

The following code stores the model in watsonx.ai / WML and retrieves its model asset ID. #thampp: The ids are unique only within the container.


In [22]:
print("Storing model .....")

published_model_details = watsonx_ai_client.repository.store_model(model=classifier, meta_props=model_props, training_data=x_train, training_target=y_train)
model_id = watsonx_ai_client.repository.get_model_id(published_model_details)
print("Done")
print("Model ID: {}".format(model_id))

Storing model .....
Done
Model ID: b3901ee7-1133-46a5-9fa8-5af81340bdfc


---
## Retrieve Saved Model with Factsheet Client

The model, saved using watsonx.ai / WML methods, includes comprehensive training documentation in the factsheet. 
Although additional information can be manually added, such as diagrams, this is beyond the scope of this notebook.


For more information, refer to the [documentation](https://s3.us.cloud-object-storage.appdomain.cloud/aifactsheets-client/doc_files/AIGov/Retrive%20Model.html).


To associate the model with an AI use case, retrieve it using the `assets.get_model()` method:

- Use `verbose=True` for detailed information.
- Retrieve the model by `model_id` with:
  - facts_client.assets.`get_model(model_id=<wml_model_id>)`
  
  - facts_client.assets.`get_model(model_id=<wml_model_id>, container_type=<space_or_project>, container_id=<space_or_project_id>)`


In [23]:
watsonx_ai_model=facts_client.assets.get_model(wml_stored_model_details=published_model_details)
watsonx_ai_model.get_info(verbose=True)

2026/03/19 12:52:53 INFO : 
Current model Information:
  Asset ID       : b3901ee7-1133-46a5-9fa8-5af81340bdfc
  Container Type : project
  Container ID   : eb58b8b3-2fd3-4702-944b-12299ec8424d
  Facts Type     : modelfacts_user



{'name': 'IrisScikitModel',
 'asset_type': 'wml_model',
 'url': 'https://cpd-cpd-instance.apps.fsqa-530-497.cp.fyre.ibm.com/ml/models/b3901ee7-1133-46a5-9fa8-5af81340bdfc?projectid=eb58b8b3-2fd3-4702-944b-12299ec8424d&context=cpdaas',
 'asset_id': 'b3901ee7-1133-46a5-9fa8-5af81340bdfc',
 'container_type': 'project',
 'container_id': 'eb58b8b3-2fd3-4702-944b-12299ec8424d',
 'facts_type': 'modelfacts_user'}

In [24]:
model_ui_url = watsonx_ai_model.get_info(verbose=True)["url"]
display(Markdown("[Click here to see the created model asset and it's factsheet in the UI](" + model_ui_url + ")"))

[Click here to see the created model asset and it's factsheet in the UI](https://cpd-cpd-instance.apps.fsqa-530-497.cp.fyre.ibm.com/ml/models/b3901ee7-1133-46a5-9fa8-5af81340bdfc?projectid=eb58b8b3-2fd3-4702-944b-12299ec8424d&context=cpdaas)

---
## Inventory Management

This section focuses on the creation and management of inventory on the Watsonx.Governance platform.
The inventory is a view where you can define an AI use case to request a new model, and then track the model and related assets through its lifecycle


For more information, refer to the [Inventory Managment documentation](https://s3.us.cloud-object-storage.appdomain.cloud/aifactsheets-client/doc_files/asset_model/Model%20Usecase/Inventory.html).

In [26]:
inventory=facts_client.assets.create_inventory(name="New Inventory",description="testing")

-----------------------------------------------------------------------------------------------------------------------------
                                                  Inventory Creation Started                                                 
-----------------------------------------------------------------------------------------------------------------------------
2026/03/19 12:53:39 INFO : Inventory 'New Inventory' has been created successfully.
2026/03/19 12:53:39 INFO : Details:
  Inventory ID: df8c3763-58f9-4ee7-9316-6a0e09031eed
  Inventory Name: New Inventory
  Description: testing
  Creator ID: 1000331001
  Creator Name: cpadmin



In [27]:
inventory.get_info()

{'inventory_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed',
 'name': 'New Inventory',
 'description': 'testing',
 'creator': ('cpadmin', None),
 'creator_id': '1000331001'}

---

<h2>Inventory Collaborators</h2>

Inventories are inherently collaborative, allowing multiple users with distinct roles to participate in the inventory.

This section provides a detailed overview of the methods available for managing collaborators within each inventory item, including:

- Viewing the current list of collaborators
- Assigning new collaborators
- Defining roles for each collaborator
- Removing existing collaborators




<h3 style="color:gold;">Retrieve a list of collaborators for the inventory. </h3>

In [28]:
# Get the list of collaborators
inventory.list_collaborators()

-----------------------------------------------------------------------------------------------------------------------------
                                                Collaborator Retrieval Started                                               
-----------------------------------------------------------------------------------------------------------------------------
2026/03/19 12:53:49 INFO : Collaborators fetched successfully


[{'name': ('cpadmin', None),
  'user_id': '1000331001',
  'role': 'admin',
  'created_time': '2026-03-19T07:23:36Z'}]



<h3 style="color:gold;">Assigning New Collaborators</h3>

To add new collaborators, you need either the `user_id` or the `access_group_id`, depending on the platform you're using.

#### In the Watsonx.Gov Platform:
You need either the `user_id` or the `access_group_id`. To find these:

1. **Find the `user_id`:**
   - Go to the Watsonx.Gov Platform.
   - Navigate to **Access Control**.
   - Select **Users**.
   - Locate the `User-ID` for the desired user.

2. **Find the `access_group_id`:**
   - Go to the Watsonx.Gov Platform.
   - Navigate to **Access Control**.
   - Select **Users Group**.
   - Find the desired group.
   - The `access_group_id` is located at the end of the URL, e.g., `usermgmt-ui/groups/10001`, where `10001` is the group ID.




In [ ]:
# Determine user_id and access_group_id based on use_software flag

user_id = "1000000**"
access_group_id = "1000*"



In [ ]:
if user_id:
    inventory.add_collaborator(user_id=user_id, role="editor")
elif access_group_id:
    inventory.add_collaborator(access_group_id=access_group_id, role="editor")
else:
    raise ValueError("Either user_id or access_group_id must be provided")



---
## Creation of New AI Use Case <a href="#add_mut"></a>

An **AI Use Case** tracks model asset lifecycles across environments like development, pre-production, and production. 

**Note:** The term "AI Use Case" has replaced "Model Use Case" to reflect a broader range of AI assets. While some APIs may still use the old terminology, it will be phased out.


For more information, refer to the [New AI Use Case](https://s3.us.cloud-object-storage.appdomain.cloud/aifactsheets-client/doc_files/asset_model/Model%20Usecase/Creation%20of%20Model%20Uscase.html).


- If `ai_usecase_id` is not provided, the default inventory_id is used (requires `EDITOR` access).

- Retrieve the AI Use Case ID from the URL in inventory or by using `get_ai_usecase()`.

- For Cloud Pak for Data, ensure OpenPages integration is disabled (create inventory permission needed).


In [29]:
ai_usecase_inventory_id = inventory.get_id()
ai_usecase_name="Automatic Iris classification - demonstration use case" 
ai_usecase_desc="AI usecase for iris classification"

In [30]:
print(ai_usecase_inventory_id)

df8c3763-58f9-4ee7-9316-6a0e09031eed


##### ⚠️ Attention: Use of inventories vs. catalogs as input to the `catalog_id` parameter

- The `catalog_id` parameter can specify either an inventory ID or a catalog ID. Inventories which technically are a sub-type of catalogs optimized for watsonx.governance, are recommended.


- Catalogs are still supported but will be deprecated over time. As a best practice, use inventories for storing use cases and external models.


In [31]:
ai_usecase = facts_client.assets.create_ai_usecase(catalog_id=ai_usecase_inventory_id,name=ai_usecase_name,description=ai_usecase_desc)
ai_usecase.get_info(True)

2026/03/19 13:26:39 INFO : AI usecase created successfully


{'ai_usecase_name': 'Automatic Iris classification - demonstration use case',
 'description': 'AI usecase for iris classification',
 'asset_type': 'model_entry',
 'url': 'https://cpd-cpd-instance.apps.fsqa-530-497.cp.fyre.ibm.com/data/catalogs/df8c3763-58f9-4ee7-9316-6a0e09031eed/asset/d1c51a44-2069-477e-b2ec-3bc8b2e16bf6?context=cpdaas',
 'model_usecase_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6',
 'container_type': 'catalog',
 'catalog_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed',
 'facts_type': 'model_entry_user'}

#### Methods Available for AI Use Cases

Explore the following methods for managing AI Use Cases:

In [32]:
print("AI usecase name is : {}".format(ai_usecase.get_name()))
print("AI usecase ID is : {}".format(ai_usecase.get_id()))
print("AI usecase catalog is : {}".format(ai_usecase.get_container_id()))
print("AI usecase container type is : {}".format(ai_usecase.get_container_type()))
print("AI usecase description is : {}".format(ai_usecase.get_description()))

AI usecase name is : Automatic Iris classification - demonstration use case
AI usecase ID is : d1c51a44-2069-477e-b2ec-3bc8b2e16bf6
AI usecase catalog is : df8c3763-58f9-4ee7-9316-6a0e09031eed
AI usecase container type is : catalog
AI usecase description is : AI usecase for iris classification


---
## Create an Approach <a href="#createapproach"></a>

- Track multiple models and prompts under a single use case by grouping them into different approaches.
- Create multiple approaches for various classification algorithms to facilitate comparison and integration.
- Use approaches to manage different models that need to be combined for a specific use case.



For more information, refer to the [Approach Documentation](https://s3.us.cloud-object-storage.appdomain.cloud/aifactsheets-client/doc_files/asset_model/Model%20Usecase/Model%20Usecase%20Approach.html)



#### Create an Approach for Decision Tree classification
- Define a new approach specifically for Decision Tree classification within the existing use case.
- This approach will enable tracking and management of the Decision Tree model alongside other classification algorithms.

In [33]:
decisiontree_approach = ai_usecase.create_approach(name="Decision Tree classification",description="Use a descision tree approach to classify iris data",icon="Sprout",color="Teal")
decisiontree_approach.get_info()

2026/03/19 13:26:58 INFO : Approach created successfully


{'approach_id': '553272b1-7b38-4096-97fc-a0ef440d8182',
 'approach_name': 'Decision Tree classification',
 'approach_desc': 'Use a descision tree approach to classify iris data',
 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6',
 'model_container_type': 'catalog',
 'model_container_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed'}

#### Create an Approach for Random Forest Classification

- Define a new approach specifically for Random Forest classification within the existing use case.
- This approach will enable tracking and management of the Random Forest model alongside other classification algorithms.


In [34]:
randomforest_approach = ai_usecase.create_approach(name="Random Forest classification",description="Use a Random Forest approach to classify iris data",icon="Tree",color="Green")
randomforest_approach.get_info()

2026/03/19 13:27:01 INFO : Approach created successfully


{'approach_id': 'eec98acf-61c7-4cb2-b1f5-8f26f8889489',
 'approach_name': 'Random Forest classification',
 'approach_desc': 'Use a Random Forest approach to classify iris data',
 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6',
 'model_container_type': 'catalog',
 'model_container_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed'}

### Retrieve All Approaches

- Fetch a list of all approaches associated with the current use case.
- This allows you to review and manage the various models and methods tracked under the use case.


In [35]:
approaches = ai_usecase.get_approaches()
for approach_detail in approaches:
    print(approach_detail.get_info())

2026/03/19 13:27:03 INFO : Approaches retrieved successfully
{'approach_id': 'eec98acf-61c7-4cb2-b1f5-8f26f8889489', 'approach_name': 'Random Forest classification', 'approach_desc': 'Use a Random Forest approach to classify iris data', 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6', 'model_container_type': 'catalog', 'model_container_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed'}
{'approach_id': '553272b1-7b38-4096-97fc-a0ef440d8182', 'approach_name': 'Decision Tree classification', 'approach_desc': 'Use a descision tree approach to classify iris data', 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6', 'model_container_type': 'catalog', 'model_container_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed'}
{'approach_id': '00000000-0000-0000-0000-000000000000', 'approach_name': 'Default approach', 'approach_desc': 'A default approach for tracking your AI assets.', 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6', 'model_container_type': 'catalog', 'model_container_id': 

### Retrieve Single Approach

- Retrieve details of a specific approach within the use case.
- This allows you to access information about a particular model or method tracked under the use case.


In [36]:
decisiontree_approach = ai_usecase.get_approach(approach_id=approaches[1].get_id())
decisiontree_approach.get_info()

2026/03/19 13:27:05 INFO : Approach retrieved successfully


{'approach_id': '553272b1-7b38-4096-97fc-a0ef440d8182',
 'approach_name': 'Decision Tree classification',
 'approach_desc': 'Use a descision tree approach to classify iris data',
 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6',
 'model_container_type': 'catalog',
 'model_container_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed'}

> Here, you can see that there are three approaches: two are user-created, and one is the default approach. The default approach cannot be deleted. **Note:** We can Modify the default approach to **"Custom Approach"** as per user requirements and use it accordingly. The original default approach itself cannot be deleted.


In [37]:
default_approach = ai_usecase.get_approach(approach_id=approaches[2].get_id())
default_approach.get_info()

2026/03/19 13:27:07 INFO : Approach retrieved successfully


{'approach_id': '00000000-0000-0000-0000-000000000000',
 'approach_name': 'Default approach',
 'approach_desc': 'A default approach for tracking your AI assets.',
 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6',
 'model_container_type': 'catalog',
 'model_container_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed'}

In [38]:
default_approach.set_name(name="Model Regression classicifation")
default_approach.set_description(description="Use a Regression  tree approach to classify iris data")

default_approach.get_info()

2026/03/19 13:27:12 INFO : Approach name changed successfully
2026/03/19 13:27:15 INFO : Approach description changed successfully


{'approach_id': '00000000-0000-0000-0000-000000000000',
 'approach_name': 'Default approach',
 'approach_desc': 'A default approach for tracking your AI assets.',
 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6',
 'model_container_type': 'catalog',
 'model_container_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed'}

### Delete an Approach

- Remove a specific approach from the use case.
- This action deletes the associated model or method, helping to manage and streamline your tracking process.


In [39]:
ai_usecase.remove_approach(approach=approaches[0])

2026/03/19 13:27:24 INFO : Approach removed successfully


### Other Methods available for approach
Explore the following methods for managing approach:

In [40]:
approaches = ai_usecase.get_approaches()
decisiontree_approach = ai_usecase.get_approach(approach_id=approaches[0].get_id())
print(decisiontree_approach)

## Update approach name and description
decisiontree_approach.set_name(name="Decision Tree classification")
decisiontree_approach.set_description(description="Use a descision tree approach to classify iris data")

## Get approach versions
decisiontree_approach.get_versions()

## Get approach info
decisiontree_approach.get_info()

2026/03/19 13:27:26 INFO : Approaches retrieved successfully
2026/03/19 13:27:28 INFO : Approach retrieved successfully
{
  "approach_id": "553272b1-7b38-4096-97fc-a0ef440d8182",
  "approach_name": "Decision Tree classification",
  "approach_desc": "Use a descision tree approach to classify iris data",
  "model_asset_id": "d1c51a44-2069-477e-b2ec-3bc8b2e16bf6",
  "model_container_type": "catalog",
  "model_container_id": "df8c3763-58f9-4ee7-9316-6a0e09031eed"
}
2026/03/19 13:27:30 INFO : Approach name changed successfully
2026/03/19 13:27:32 INFO : Approach description changed successfully


{'approach_id': '553272b1-7b38-4096-97fc-a0ef440d8182',
 'approach_name': 'Decision Tree classification',
 'approach_desc': 'Use a descision tree approach to classify iris data',
 'model_asset_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6',
 'model_container_type': 'catalog',
 'model_container_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed'}

In [41]:
print("Approach name is : {}".format(decisiontree_approach.get_name()))
print("Approach ID is : {}".format(decisiontree_approach.get_id()))
print("Approach description is : {}".format(decisiontree_approach.get_description()))
print("Approach usecase is : {}".format(decisiontree_approach.get_model_useacse_id()))
print("Approach usecase container type is : {}".format(decisiontree_approach.get_model_usecase_container_type()))
print("Approach usecase container ID is : {}".format(decisiontree_approach.get_model_usecase_container_id()))

Approach name is : Decision Tree classification
Approach ID is : 553272b1-7b38-4096-97fc-a0ef440d8182
Approach description is : Use a descision tree approach to classify iris data
Approach usecase is : d1c51a44-2069-477e-b2ec-3bc8b2e16bf6
Approach usecase container type is : catalog
Approach usecase container ID is : df8c3763-58f9-4ee7-9316-6a0e09031eed


---
## Associated Workspace

Associate workspacesis employed to associate AI use cases with the appropriate workspaces, ensuring they are organized around the same business problem. This involves setting up workspaces that will be used throughout the development process to tackle the business problem or implement the use case smoothly.


For more information, refer to the [Associate Workspaces](https://s3.us.cloud-object-storage.appdomain.cloud/aifactsheets-client/doc_files/asset_model/Model%20Usecase/Associate%20Workspaces.html).


### Rules for Associating Workspaces

- A project can be associated with just one AI use case. Multiple projects might contribute assets or validations to a single AI use case, but each project is restricted to associating with a single-use case.

- A project can be associated with a single lifecycle phase for an AI use case. That is, you cannot use the same project to develop and validate an asset.

- A space can be associated with multiple AI use cases. That is, if you use a single space for deployed models in production, assets in that space might be tracked in different AI use cases. This distinction from how projects relate to an AI use case reflects how spaces are often used in ModelOps to manage a collection of deployments in a particular state.

`The following graphic illustrates the relationship of associated workspaces with an AI use case.`


<p align="center">
  <img src="https://dataplatform.cloud.ibm.com/docs/api/content/wsj/analyze-data/images/xgov-associate-workspace.svg?context=wx&locale=en" alt="Associate Workspace" style="width: 40%; max-width: 400px;">
</p>




### Retrieve All Associated Workspaces for the AI Use Case




In [42]:
ai_usecase.list_all_associated_workspaces()

model_usecase_id :d1c51a44-2069-477e-b2ec-3bc8b2e16bf6 ,catalog_id : df8c3763-58f9-4ee7-9316-6a0e09031eed 
2026/03/19 13:27:39 INFO : Associated Workspaces retrieved successfully for the given AI usecase with id: d1c51a44-2069-477e-b2ec-3bc8b2e16bf6
2026/03/19 13:27:39 INFO : No associated workspaces found for the specified model_usecase_id: d1c51a44-2069-477e-b2ec-3bc8b2e16bf6


#### Add Project Workspace to AI Use Case

This feature allows you to link a project workspace to an AI use case, ensuring that all related tasks, data, and documentation are organized together for efficient development and management.


In [43]:
# add project workspace to model usecase
ai_usecase.add_workspaces_associations(workspace_id=container_id, workspace_type="project", phase_name="develop")

model_usecase_id :d1c51a44-2069-477e-b2ec-3bc8b2e16bf6 ,catalog_id : df8c3763-58f9-4ee7-9316-6a0e09031eed 
2026/03/19 13:27:58 INFO : Associated Workspaces retrieved successfully for the given AI usecase with id: d1c51a44-2069-477e-b2ec-3bc8b2e16bf6
2026/03/19 13:28:07 INFO : Workspace associations attempted for the given workspace IDs: eb58b8b3-2fd3-4702-944b-12299ec8424d
2026/03/19 13:28:07 INFO : Workspace association for model use case with id d1c51a44-2069-477e-b2ec-3bc8b2e16bf6 done successfully for the following workspace(s)==>

{'id': 'eb58b8b3-2fd3-4702-944b-12299ec8424d', 'type': 'project', 'name': 'test_16/3', 'is_deleted': False}


---
## Track a Model Under an AI Use Case <a name="add_mu"></a>

- **AI Use Cases** are designed to monitor the lifecycle of model assets across various stages, including development, pre-production, and production.
- To effectively integrate a model into an AI use case, three critical elements must be addressed: the **model**, the **AI use case**, and the **approach**.
- Link an existing AI use case by using the following method: `model.track(usecase=<ai_usecase>, approach=<approach1>, version_number="<Version Number>")`.

- **Version Numbers** are categorized as follows:
    - **Major Version:** Indicates significant changes, represented as `1.0.0`.
    - **Minor Version:** Reflects incremental improvements, represented as `0.1.0`.
    - **Patch Version:** Denotes minor fixes or updates, represented as `0.0.1`.
    - **Custom Version:** Allows for tailored versioning according to specific user needs.

- Ensure that the `ai_usecase`, `approach`, and `version_number` parameters are provided as mandatory.


For more information, refer to the [Governing AI Assets](https://s3.us.cloud-object-storage.appdomain.cloud/aifactsheets-client/doc_files/AIGov/Tracking%20models%20with%20AI%20Factsheets.html)


In [44]:
watsonx_ai_model.track(usecase=ai_usecase,approach=decisiontree_approach,version_number="major",version_comment="major update to previous version")

-----------------------------------------------------------------------------------------------------------------------------
                                                   Tracking Process Started                                                  
-----------------------------------------------------------------------------------------------------------------------------
2026/03/19 13:28:35 INFO : Assigned Decision Tree classification to Automatic Iris classification - demonstration use case for tracking.
2026/03/19 13:28:39 INFO : Initiate linking model to existing AI usecase d1c51a44-2069-477e-b2ec-3bc8b2e16bf6
2026/03/19 13:28:49 INFO : Successfully finished linking Model b3901ee7-1133-46a5-9fa8-5af81340bdfc to AI usecase


{'model_entry_catalog_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed',
 'model_entry_id': 'd1c51a44-2069-477e-b2ec-3bc8b2e16bf6',
 'model_entry_name': 'Automatic Iris classification - demonstration use case',
 'master_copy_id': '22f590a3-c9b2-486c-a134-c10ea341c2b9',
 'master_copy_inventory_id': 'df8c3763-58f9-4ee7-9316-6a0e09031eed',
 'model_entry_status': 'Draft',
 'version_number': '1.0.0',
 'version_comment': 'major update to previous version',
 'approach_name': 'Decision Tree classification'}

#### Retrieve Tracked Models for Use Case

- Fetch all models that are currently tracked under a specified AI use case.
- This provides an overview of all models associated with the use case, facilitating management and analysis.


In [45]:
ai_usecase.get_tracked_models()

[{'id': 'b3901ee7-1133-46a5-9fa8-5af81340bdfc',
  'name': 'IrisScikitModel',
  'type': 'wml_model',
  'container_id': 'eb58b8b3-2fd3-4702-944b-12299ec8424d',
  'container_name': 'test_16/3',
  'container_type': 'project',
  'is_deleted': False,
  'master_id': '00b913af-b8a6-4ca0-9347-1c64bac71104',
  'model_identity_key': '00b913af-b8a6-4ca0-9347-1c64bac71104',
  'source_asset_id': '',
  'algorithm': '',
  'phase_name': 'Develop',
  'deployment_space_type': 'development',
  'openpages_details': {},
  'version_details': {'number': '1.0.0',
   'approach_id': '553272b1-7b38-4096-97fc-a0ef440d8182',
   'comment': 'major update to previous version'},
  'externally_managed': False,
  'deployments': []}]

## Untrack a Model

Remove a model from an AI use case when it is no longer relevant or needs to be managed separately.


In [46]:
watsonx_ai_model.untrack()

2026/03/19 13:29:03 INFO : Starting to untrack the model asset b3901ee7-1133-46a5-9fa8-5af81340bdfc from the AI use case d1c51a44-2069-477e-b2ec-3bc8b2e16bf6
2026/03/19 13:29:07 INFO : Successfully completed untracking of Watsonx Governance model asset b3901ee7-1133-46a5-9fa8-5af81340bdfc from the AI use case.


---
## Cleanup<a href="#clean"></a>

In [47]:
if run_cleanup_at_end:
    facts_client.assets.remove_asset(asset_id=ai_usecase.get_info()["model_usecase_id"],container_type=ai_usecase.get_info()["container_type"],container_id=ai_usecase.get_info()["catalog_id"])
    facts_client.assets.remove_asset(asset_id=watsonx_ai_model.get_info()["asset_id"],container_type=watsonx_ai_model.get_info()["container_type"],container_id=watsonx_ai_model.get_info()["container_id"])
else:
    model_ui_url = watsonx_ai_model.get_info(verbose=True)["url"]
    display(Markdown("[Click here to see the created wml model details in the UI](" + model_ui_url + ")"))
    ai_usecase_ui_url = ai_usecase.get_info(verbose=True)["url"]
    display(Markdown("[Click here to see the created AI use case in the UI](" + ai_usecase_ui_url + ")"))

2026/03/19 13:29:13 INFO : Successfully deleted asset id d1c51a44-2069-477e-b2ec-3bc8b2e16bf6 in catalog df8c3763-58f9-4ee7-9316-6a0e09031eed
2026/03/19 13:29:17 INFO : Successfully deleted asset id b3901ee7-1133-46a5-9fa8-5af81340bdfc in project eb58b8b3-2fd3-4702-944b-12299ec8424d



### Delete Inventory

This section covers the process for deleting an inventory item. Make sure to carefully review the item before proceeding, as deletion is irreversible.


In [ ]:
inventory.delete_inventory()

**Created by:**  


IBM watsonx.governance - *AI Factsheet Python SDK Team*

---

**Copyright © 2022-2024 IBM**  
Released under the [Apache 2.0 License](https://www.apache.org/licenses/LICENSE-2.0).
